In [1]:
# Medora Fine-Tuning — Kaggle Notebook
# Run on Kaggle with GPU T4 x2 (free tier)
# Trains Gemma 4 E4B on the Medora medication counseling dataset

In [2]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 26.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 84.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 40.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.9 MB/s eta 0:00

In [3]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
model_name="unsloth/gemma-4-E2B-it",
max_seq_length=2048,
load_in_4bit=True,
full_finetuning=False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

In [4]:
model = FastModel.get_peft_model(
model,
r=32,
target_modules=[
"q_proj", "k_proj", "v_proj", "o_proj",
"gate_proj", "up_proj", "down_proj",
],
lora_alpha=64,
lora_dropout=0,
use_gradient_checkpointing="unsloth",
random_state=42,
)

[unsloth_zoo.log|WARNING]Unsloth: Failed to register input-embedding hook for `model.base_model.model.model.audio_tower`: `get_input_embeddings` not auto‑handled for Gemma4AudioModel; please override in the subclass.. Falling back to pre-forward hook.


In [5]:
from datasets import load_dataset
dataset = load_dataset(
"json",
data_files="/kaggle/input/datasets/wallfou/medora-training-data2/medora_train.jsonl",
split="train",
)


# Validating dataset shape
print(f"Total training examples: {len(dataset)}")
for i, row in enumerate(dataset):
    msgs = row.get("messages", [])
    if len(msgs) < 2:
        print(f"WARNING: Row {i} has only {len(msgs)} messages")
    if msgs[0]["role"] != "system":
        print(f"WARNING: Row {i} first message is not system role")

print(f"Sample question: {dataset[0]['messages'][1]['content'][:80]}...")
print(f"Sample answer: {dataset[0]['messages'][2]['content'][:80]}...")

Generating train split: 0 examples [00:00, ? examples/s]

Total training examples: 137
Sample question: What are the side effects of warfarin?...
Sample answer: Warfarin's most common side effect is bleeding more easily than usual. You might...


In [6]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-4",
)

def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_example)

# Split 90/10 for train/eval to detect overfitting
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]
print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

Map:   0%|          | 0/137 [00:00<?, ? examples/s]

Train: 123, Eval: 14


In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=6,
        learning_rate=1e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        eval_strategy="steps",
        eval_steps=20,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

# Detect which turn tokens the template actually uses
sample_text = train_dataset[0]["text"]
if "<start_of_turn>user" in sample_text:
    inst_token = "<start_of_turn>user\n"
    resp_token = "<start_of_turn>model\n"
    print("Detected: <start_of_turn> style tokens")
elif "<|turn>user" in sample_text:
    inst_token = "<|turn>user\n"
    resp_token = "<|turn>model\n"
    print("Detected: <|turn> style tokens")
else:
    raise ValueError(
        f"Could not detect turn tokens in formatted text. "
        f"First 500 chars:\n{sample_text[:500]}"
    )

# Apply response only training
trainer = train_on_responses_only(
    trainer,
    instruction_part=inst_token,
    response_part=resp_token,
)

# Verify masking actually worked
sample = trainer.train_dataset[0]
labels = sample["labels"]
masked = sum(1 for l in labels if l == -100)
total = len(labels)
pct = masked / total
print(f"Masked {masked}/{total} tokens ({pct:.0%})")
if pct < 0.2 or pct > 0.95:
    raise ValueError(
        f"Masking looks wrong ({pct:.0%})."
    )
print("Response only masking verified.")

print("\nStarting training...")
trainer_stats = trainer.train()
print(f"Training time: {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"Final train loss: {trainer_stats.metrics['train_loss']:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/123 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/14 [00:00<?, ? examples/s]

Detected: <|turn> style tokens


Map (num_proc=8):   0%|          | 0/123 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/123 [00:00<?, ? examples/s]

...

Masked 82/220 tokens (37%)
Response only masking verified.

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 123 | Num Epochs = 6 | Total steps = 96
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 62,078,976 of 5,185,256,992 (1.20% trained)


Step,Training Loss,Validation Loss
20,0.382390,1.644698
40,0.298029,1.504314
60,0.241578,1.495264
80,0.433724,1.559372
96,0.355368,1.584920


Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-96/tokenizer_config.json.


Training time: 298s
Final train loss: 0.3455


In [8]:
from transformers import AutoTokenizer
text_tokenizer = AutoTokenizer.from_pretrained("unsloth/gemma-4-E2B-it")

def run_test(messages, max_tokens=300):
    text = text_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = text_tokenizer(text, return_tensors="pt").to("cuda")
    from transformers import TextStreamer
    streamer = TextStreamer(text_tokenizer, skip_prompt=True)
    _ = model.generate(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_tokens,
        temperature=0.3,
        top_p=0.9,
    )

print("\nTest 1: Side effects of warfarin: ")
run_test([
    {"role": "system", "content": "You are Medora, a warm and knowledgeable medication safety assistant. The patient takes these medications: warfarin. Use short, simple sentences. Explain medical terms in plain language. Share standard patient education information freely. Defer diagnosis and dosage decisions to doctors. Never repeat the same disclaimer twice in a row."},
    {"role": "user", "content": "What are the side effects of warfarin?"},
])

print("\nTest 2: Off-topic handling: ")
run_test([
    {"role": "system", "content": "You are Medora, a warm and knowledgeable medication safety assistant. The patient takes these medications: warfarin. Use short, simple sentences. Explain medical terms in plain language. Share standard patient education information freely. Defer diagnosis and dosage decisions to doctors. Never repeat the same disclaimer twice in a row."},
    {"role": "user", "content": "How do I use recursion to sort an array?"},
], max_tokens=100)

print("\nTest 3: Empathy: ")
run_test([
    {"role": "system", "content": "You are Medora, a warm and knowledgeable medication safety assistant. The patient takes these medications: warfarin. Use short, simple sentences. Explain medical terms in plain language. Share standard patient education information freely. Defer diagnosis and dosage decisions to doctors. Never repeat the same disclaimer twice in a row."},
    {"role": "user", "content": "I'm worried I'm addicted to oxycodone. What should I do?"},
])

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]


Test 1: Side effects of warfarin: 
Warfarin prevents blood clots, which is great for preventing strokes and heart attacks. But it's a "narrow-striped" medication — meaning the difference between an effective dose and a dangerous one is smaller than with other drugs.

The most common side effect is bleeding more easily than usual. Watch for small cuts taking longer to stop, nosebleeds, or blood in urine or stool.

A key thing to know: warfarin interacts with many other drugs. Aspirin, ibuprofen, amoxicillin, and most over-the-counter pain relievers can all increase your bleeding risk. Always tell your doctor and pharmacist everything you take.

Also, warfarin affects your vitamin K levels, so eating consistent amounts of green leafy vegetables is important.<turn|>

Test 2: Off-topic handling: 
That's a programming question — I only know about medications. Is there anything about your warfarin or other prescriptions I can help with?<turn|>

Test 3: Empathy: 
I'm really sorry you're goin

In [9]:
model.save_pretrained("medora-lora")
tokenizer.save_pretrained("medora-lora")
print("LoRA adapters saved to medora-lora/")

Unsloth: Restored added_tokens_decoder metadata in medora-lora/tokenizer_config.json.


LoRA adapters saved to medora-lora/


In [15]:
# !pip install -U huggingface_hub

# import os
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import login

# user_secrets = UserSecretsClient()
# hf_token = user_secrets.get_secret("HF_TOKEN")

# login(token=hf_token)

# hf_username = "Wallfou"
# repo_name = "medora-gemma-4-lora"
# repo_id = f"{hf_username}/{repo_name}"

# model.push_to_hub(repo_id, token=hf_token)
# tokenizer.push_to_hub(repo_id, token=hf_token)